# Redrob Hackathon: Job Description Compass

## What this notebook does
This notebook reads the job description and converts it into a structured hiring rubric.

## Why we are doing this
The candidate-ranking system should not rely on keyword matching alone.  
We need a clear understanding of what the role actually wants:
- must-have skills
- preferred skills
- disqualifiers
- behavioral expectations
- location and availability preferences

This notebook turns the JD into a machine-readable compass that later notebooks can use for ranking candidates.

In [10]:
!pip install -q python-docx pandas
# python-docx is used to read the job description from the docx file.
# pandas is used to organize the extracted rubric into a readable table.

In [11]:
from pathlib import Path
from docx import Document
import json
import pandas as pd

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
# Set project paths
PROJECT_DIR = Path("/content/drive/MyDrive/Redrob_Hackathon")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
JD_PATH = DATA_DIR / "job_description.docx"

print("JD exists:", JD_PATH.exists())
print("JD path:", JD_PATH)

JD exists: True
JD path: /content/drive/MyDrive/Redrob_Hackathon/data/job_description.docx


In [15]:
# We will be using the jd to under the roles required by the company and organise
# our efforts to find out signals that matter the most in ranking

document = Document(JD_PATH)

jd_paragraphs = []
for paragraph in document.paragraphs:
    text = paragraph.text.strip()
    if text:
        jd_paragraphs.append(text)

jd_text = "\n".join(jd_paragraphs)

print(jd_text[:500])

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)
Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineer


In [17]:
# creating a clean copy of the jd so that it can be used later
jd_text_path = ARTIFACTS_DIR / "job_description_cleaned.txt"
jd_text_path.write_text(jd_text, encoding="utf-8")

print("Saved:", jd_text_path)

Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/job_description_cleaned.txt


In [18]:
jd_compass = {
    "role_title": "Senior AI Engineer",

    "role_goal": (
        "Build and ship practical AI systems involving retrieval, ranking, "
        "LLMs, evaluation, and production engineering."
    ),

    "must_have_skills": [
        "python",
        "machine learning",
        "llm",
        "embeddings",
        "retrieval",
        "vector search",
        "ranking",
        "evaluation",
    ],

    "strong_preferred_skills": [
        "rag",
        "fine tuning",
        "transformers",
        "pytorch",
        "faiss",
        "elasticsearch",
        "docker",
        "aws",
        "gcp",
        "langchain",
        "langgraph",
    ],

    "experience_preferences": {
        "minimum_years": 3,
        "ideal_years_min": 4,
        "ideal_years_max": 10,
        "production_experience_required": True,
        "recent_production_code_preferred": True,
    },

    "behavioral_preferences": {
        "open_to_work_bonus": True,
        "recent_activity_bonus": True,
        "high_response_rate_bonus": True,
        "short_notice_period_bonus": True,
    },

    "negative_signals": [
        "only_recent_langchain_or_openai_experience",
        "no_recent_production_code",
        "keyword_stuffing",
        "career_timeline_inconsistency",
    ],
}

In [19]:
for key, value in jd_compass.items():
    print(f"\n{'=' * 80}")
    print(key.upper())
    print("=" * 80)
    print(value)


ROLE_TITLE
Senior AI Engineer

ROLE_GOAL
Build and ship practical AI systems involving retrieval, ranking, LLMs, evaluation, and production engineering.

MUST_HAVE_SKILLS
['python', 'machine learning', 'llm', 'embeddings', 'retrieval', 'vector search', 'ranking', 'evaluation']

STRONG_PREFERRED_SKILLS
['rag', 'fine tuning', 'transformers', 'pytorch', 'faiss', 'elasticsearch', 'docker', 'aws', 'gcp', 'langchain', 'langgraph']

EXPERIENCE_PREFERENCES
{'minimum_years': 3, 'ideal_years_min': 4, 'ideal_years_max': 10, 'production_experience_required': True, 'recent_production_code_preferred': True}

BEHAVIORAL_PREFERENCES
{'open_to_work_bonus': True, 'recent_activity_bonus': True, 'high_response_rate_bonus': True, 'short_notice_period_bonus': True}

NEGATIVE_SIGNALS
['only_recent_langchain_or_openai_experience', 'no_recent_production_code', 'keyword_stuffing', 'career_timeline_inconsistency']


In [20]:
skill_groups = {
    "core_ai_ml": [
        "machine learning",
        "deep learning",
        "llm",
        "large language model",
        "transformers",
        "pytorch",
        "tensorflow",
    ],

    "retrieval_ranking": [
        "embeddings",
        "retrieval",
        "vector search",
        "vector database",
        "faiss",
        "elasticsearch",
        "bm25",
        "ranking",
        "reranking",
        "semantic search",
        "rag",
    ],

    "production_engineering": [
        "python",
        "docker",
        "aws",
        "gcp",
        "azure",
        "api",
        "fastapi",
        "deployment",
        "production",
        "mlops",
    ],

    "evaluation_and_quality": [
        "evaluation",
        "benchmarking",
        "testing",
        "monitoring",
        "experimentation",
        "ab testing",
        "metrics",
    ],
}

In [21]:
for group_name, skills in skill_groups.items():
    print(f"\n{group_name.upper()}")
    for skill in skills:
        print("-", skill)


CORE_AI_ML
- machine learning
- deep learning
- llm
- large language model
- transformers
- pytorch
- tensorflow

RETRIEVAL_RANKING
- embeddings
- retrieval
- vector search
- vector database
- faiss
- elasticsearch
- bm25
- ranking
- reranking
- semantic search
- rag

PRODUCTION_ENGINEERING
- python
- docker
- aws
- gcp
- azure
- api
- fastapi
- deployment
- production
- mlops

EVALUATION_AND_QUALITY
- evaluation
- benchmarking
- testing
- monitoring
- experimentation
- ab testing
- metrics


In [22]:
scoring_dimensions = {
    "semantic_role_fit": {
        "description": "How closely the candidate profile text matches the Senior AI Engineer role.",
        "weight": 0.30,
    },

    "must_have_skill_match": {
        "description": "Match on core AI, ML, retrieval, ranking, and evaluation skills.",
        "weight": 0.20,
    },

    "career_and_seniority_fit": {
        "description": "Whether experience level, title progression, and role history fit a senior AI engineering role.",
        "weight": 0.15,
    },

    "production_evidence": {
        "description": "Evidence of building and shipping real systems, APIs, deployment, cloud, or MLOps work.",
        "weight": 0.15,
    },

    "behavioral_readiness": {
        "description": "Open-to-work status, activity, recruiter responsiveness, notice period, and interview completion.",
        "weight": 0.10,
    },

    "trust_and_consistency": {
        "description": "Profile completeness, career consistency, trap-risk flags, and evidence quality.",
        "weight": 0.10,
    },
}

In [23]:
total_weight = sum(dim["weight"] for dim in scoring_dimensions.values())
print("Total weight:", total_weight)

assert round(total_weight, 5) == 1.0

Total weight: 1.0


In [24]:
rubric_rows = []

for dimension_name, details in scoring_dimensions.items():
    rubric_rows.append({
        "dimension": dimension_name,
        "weight": details["weight"],
        "description": details["description"],
    })

rubric_df = pd.DataFrame(rubric_rows)
display(rubric_df)

,dimension,weight,description
0,semantic_role_fit,0.30,How closely the candidate profile text matches...
1,must_have_skill_match,0.20,"Match on core AI, ML, retrieval, ranking, and ..."
2,career_and_seniority_fit,0.15,"Whether experience level, title progression, a..."
3,production_evidence,0.15,Evidence of building and shipping real systems...
4,behavioral_readiness,0.10,"Open-to-work status, activity, recruiter respo..."
5,trust_and_consistency,0.10,"Profile completeness, career consistency, trap..."


In [25]:
jd_compass = {
    "jd_summary": jd_compass,
    "skill_groups": skill_groups,
    "scoring_dimensions": scoring_dimensions,
}

jd_compass_path = ARTIFACTS_DIR / "jd_compass.json"

with open(jd_compass_path, "w", encoding="utf-8") as f:
    json.dump(jd_compass, f, indent=4)

print("Saved:", jd_compass_path)

Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/jd_compass.json


In [26]:
files_to_check = [
    "job_description_cleaned.txt",
    "jd_compass.json",
]

for file_name in files_to_check:
    file_path = ARTIFACTS_DIR / file_name
    print(f"{file_name:<35} {'SAVED' if file_path.exists() else 'MISSING'}")

job_description_cleaned.txt         SAVED
jd_compass.json                     SAVED
